In [ ]:
!git clone https://github.com/michaelyserrano/mmai.git  # skip if continuing from 01
%cd mmai
!pip install -q -r project/requirements.txt
import sys, os
sys.path.insert(0, "project")
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import numpy as np, json
from pathlib import Path
from data.load_toolbench import load_api_corpus, load_eval_examples
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import evaluate_batch

TOOLBENCH_DIR = Path("toolbench")
corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
corpus_matrix = np.load("project/corpus_embeddings.npy")
with open("project/api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}
evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
print(f"Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples")

In [ ]:
index = build_faiss_index(corpus_matrix)
print(f"\u2713 FAISS index ready: {index.ntotal} vectors")

In [ ]:
from tqdm import tqdm
queries = [e["user_query"] for e in evals]
query_embeddings = get_embeddings(queries)
print(f"\u2713 Query embeddings: {query_embeddings.shape}")

In [ ]:
K = 10
all_retrieved = []
all_gt_indices = []

for i, ex in enumerate(evals):
    top_k_indices = retrieve_top_k(query_embeddings[i], index, k=K)
    all_retrieved.append(top_k_indices)
    gt_indices = [name_to_idx[name] for name in ex["ground_truth_apis"] if name in name_to_idx]
    all_gt_indices.append(gt_indices)

results = evaluate_batch(all_retrieved, all_gt_indices, ks=[1, 5, 10])
print("\n=== Baseline Results (Full Corpus / Random Negatives) ===")
for metric, score in results.items():
    print(f"  {metric}: {score:.4f}")

In [ ]:
import json
with open("project/results_baseline.json", "w") as f:
    json.dump({"condition": "random_full_corpus", "results": results}, f, indent=2)
print("\u2713 Saved results_baseline.json")